# Historical filtered analysis — not the current population

This snapshot used 1,652 comparisons after an edge-compound filter. It is preserved for traceability. The active `rapid_hydrograph_analysis.ipynb` now uses every one of the 2,214 distinct parent–child pairs. The previous description of this snapshot as “full population” was inaccurate.

# Equivalent Resistance vs RAPID Timing Response

This notebook tests whether changes in equivalent hydraulic resistance and collapsed-object geometry explain changes in routed hydrograph timing. It is the focused replacement for the exploratory `rapid_hydrograph_analysis_old.ipynb`.

Hydrograph inspection, event-window diagnostics, and individual curve plots belong in `hydrograph_investigation.ipynb`. Solver validation belongs in `network_variants/notebooks/equivalent_resistance_solver_investigation.ipynb`.

**Current scope:** percentage lag and absolute timing on the full 1,652-comparison plotted population. The path-feature exclusion is now resolved. The current results are immediately below; older restricted-cohort and shape investigations are retained as history.

## Current full-population lag search · 5 October 2026

**FULL-POPULATION-LAG-2026-10-05.** All **1,652 plotted comparisons / 16 network examples** now have the required geometry features. The 524 path-rich exclusions are resolved without changing metric definitions or filling missing values. All 1,128 prior enumerated records agree within the recorded tolerances. These are the current timing-search results; later restricted-cohort sections are retained as historical investigations.

The search evaluates **61 feature/model candidates**, selecting inside four grouped inner folds and assessing each held-out network example. Targets are percentage lag and absolute timing only. Whole-example grouping prevents comparisons from the same network entering training and assessment; it does not establish independence between river basins. Previous examination of these cases also means this is exploratory validation.

**Result:** the wider search does not improve percentage-lag prediction over the simpler component benchmark (nested R² .221; separate components .428). Absolute timing is more promising (nested R² .472, MAE 8.1 min; fixed selected-six R² .554, MAE 8.8 min). Model selection uses equal-example MAE, so a higher pooled R² need not win. Fixed-candidate rankings are descriptive, not an independently validated model-selection outcome.

The largest errors include recovered complete-compound collapses in examples 15 and 22. They remain in every candidate's assessment. Path-width/elongation measures still count shared edges on every path; scalable computation does not resolve their physical interpretation in large compounds.

[Full report, method, parity checks and transfer results](../validation/full_population_lag_20261005/README.md).


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
FULL_LAG_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
FULL_LAG_OUT = FULL_LAG_ROOT/'RAPID/validation/full_population_lag_20261005'
display(pd.Series(json.loads((FULL_LAG_OUT/'geometry_provenance.json').read_text())).to_frame('Coverage audit'))
display(pd.read_csv(FULL_LAG_OUT/'main_comparison.csv').round(4))
display(pd.read_csv(FULL_LAG_OUT/'equal_example_summary.csv').round(4))
display(Image(filename=str(FULL_LAG_OUT/'timing_skill.png')))
display(Image(filename=str(FULL_LAG_OUT/'held_out_timing.png')))
full_lag_choices = pd.read_csv(FULL_LAG_OUT/'selections.csv')
display(full_lag_choices.groupby(['response','feature_set','model']).size().rename('outer_folds_selected').to_frame())
display(pd.read_csv(FULL_LAG_OUT/'largest_errors.csv', dtype={'example_id':str})[['response','example_id','state_id','plot_category','observed','predicted','absolute_error']].groupby('response',sort=False).head(5).round(3))


## Current lag-only search and transfer · 5 October 2026

**FULL-POPULATION-LAG-2026-10-05.** This is the current expanded predictor comparison, using **61 candidates** and the refined timing responses. Older sections below preserve their original scope/metrics. Shape-model search is paused; those findings remain available as secondary results.

Model/feature selection uses only the **9,851 original-development comparisons**, with geometry-grouped inner validation. The selected models are frozen before testing the **3,362 original holdout comparisons**, **917 novel embedding-extension comparisons** (63 anchors excluded), and **all 1,652 real comparisons**. Extension timings use the same guarded outlet-peak interpolation and verified paired input clocks. Neither extension nor real outcomes select the synthetic model.

Percentage-lag R² is **.9966** on the original holdout, **−.929** on the wider extension and **−145.98** on real transfer. Absolute-timing R² is **.9824**, **.583** and **.133**, respectively. Despite positive real-transfer R² for absolute timing, its MAE is worse than predicting no change. Original-grid interpolation is strong; broad transfer remains unproven. Absolute timing transfers better to the embedding extension.

The real-trained, leave-network-example-out search is separate: nested R² .221 for percentage lag and .472 for absolute timing. Scores across these datasets describe different validation tasks. Hydraulic-domain limitations are unchanged by prediction accuracy.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image
TRANSFER_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
TRANSFER_OUT = TRANSFER_ROOT/'RAPID/validation/full_population_lag_20261005'
full_transfer_scores = pd.read_csv(TRANSFER_OUT/'synthetic_transfer_scores.csv')
display(pd.read_csv(TRANSFER_OUT/'synthetic_selections.csv'))
display(full_transfer_scores[full_transfer_scores.model_label.eq('selected')].round(4))
display(full_transfer_scores[full_transfer_scores.model_label.ne('selected')].round(4))
display(Image(filename=str(TRANSFER_OUT/'synthetic_transfer_skill.png')))


## Analysis contract — refreshed on 5 October 2026

All routed responses and resistance values come from `shared-head-anchored-depth-signed-inflow-20261003`, completed on 5 October. The loader verifies completed discharge artifacts, policy version, parent/child clock equality and complete resistance coverage. It includes independent, compound and **island** selections. Historical experiment trees supply only selection/topology metadata and observed-gauge references.

`Lambda_matched` remains the plotting-compatible name for the refreshed selected-domain `Lambda_K`. Length and exposure use that **same domain**. Expanded surviving-boundary comparisons are flagged; they must not be interpreted as the original nominal unit alone. Technical promotion-directory aliases remain visible in coverage but are excluded from plots. The plotted and fitted population contains one row per river, routed-child fingerprint and matched parent/child domain. Equivalent selections are recorded in an alias table. Topological labels take precedence over island labels for exact duplicates; distinct island operations remain **Island-only collapse**. This is an operation label, not a claim of a new topological class.

The three current responses are peak timing, plateau-to-plateau event-duration change and peak-excess change. The primary timing panels use guarded interpolated outlet peaks. Matching percentage and absolute-hour sections use exactly the same states, geometric predictors and filters. Sampled peaks remain in the analysis frame for resolution diagnostics; unresolved interpolation is excluded rather than replaced by sampled timing. No routing is performed by this notebook.


Validation on 5 October also corrected one case 15 connectivity error: original link ID 0 had collided with the routing no-downstream sentinel. Its routing reach ID was remapped, and the shared geometry for `S054_unit_1` / `island_15_002` was rerun without changing hydraulics or resistance. The full 2,230-geometry scan found no other collision. All 3,519 comparison rows now have valid event durations. See `RAPID/HYDRAULIC_PRODUCTION_POLICY.md` for the correction record.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "RAPID" / "src" / "rapid_tools").exists():
            return candidate
    raise FileNotFoundError("Could not locate the river-hierarchy repository root.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAPID_SRC = REPO_ROOT / "RAPID" / "src"
OUTPUT_ROOT = REPO_ROOT / "network_variants" / "output"
if str(RAPID_SRC) not in sys.path:
    sys.path.insert(0, str(RAPID_SRC))

from rapid_tools import (
    make_timing_scatter,
    supported_timing_x_metrics,
    supported_timing_y_metrics,
)
from rapid_tools.refreshed_analysis import build_refreshed_analysis_frame, DEFAULT_REFRESH_ROOT
REFRESH_ROOT = DEFAULT_REFRESH_ROOT
print(f"Refreshed results: {REFRESH_ROOT}")


## Configuration

All 16 cases completed routing and resistance refresh, including island states. `OUTPUT_ROOT` supplies static selection/gauge metadata; `REFRESH_ROOT` supplies the new numerical results. Changing only the historical output root is not sufficient.


In [ ]:
EXAMPLE_IDS = ['02', '03', '05', '11', '13','15', '21','22','37','41','45','46','48','54', '55', '56']

# matched_parent_vs_network: fraction of the full network represented by the matched parent corridor
# child_vs_matched_parent: fraction of the matched parent corridor retained by the child object
LENGTH_MODE = "matched_parent_vs_network"

# For complete closed compounds, the exposure factor is 1 because Lambda_matched
# already represents the whole compound; only nested/unit domains use routed width share.
PRIMARY_X_METRIC = "len_fraction_width_weighted_log_lambda"
# Refined outlet peaks; percentage denominator is the parent modeled lag to input peak.
PRIMARY_Y_METRIC = "interpolated_lag_shift_pct"
ABSOLUTE_Y_METRIC = "interpolated_peak_shift_hours"

# Optional sensitivity filter. Keep False for the complete result set.
EXCLUDE_NETWORK_EDGE_COMPOUNDS = True

# Expanded comparison domains are retained and flagged in domain_expanded.
# Their geometric predictors are recalculated on the same expanded domain.

## Available variables

These tables are generated from the plotting library, so the notebook stays synchronized with the implemented metrics.

In [ ]:
display(pd.Series(supported_timing_x_metrics(), name="description").rename_axis("x_metric").to_frame())
display(pd.Series(supported_timing_y_metrics(), name="description").rename_axis("timing_metric").to_frame())

## Assemble and validate the analysis frame

The coverage table should be checked before interpreting any figure. A state is usable for a particular comparison only when both its resistance predictor and selected timing response are finite.

In [ ]:
analysis_df = build_refreshed_analysis_frame(
    EXAMPLE_IDS, refresh_root=REFRESH_ROOT, geometry_root=OUTPUT_ROOT, length_mode=LENGTH_MODE)

for column in [PRIMARY_X_METRIC, PRIMARY_Y_METRIC, ABSOLUTE_Y_METRIC, "Lambda_matched"]:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors="coerce")

analysis_df["primary_pair_complete"] = (
    np.isfinite(analysis_df[PRIMARY_X_METRIC])
    & np.isfinite(analysis_df[PRIMARY_Y_METRIC])
    & np.isfinite(analysis_df[ABSOLUTE_Y_METRIC])
    & analysis_df["peak_interpolation_status"].eq("ok")
)

coverage = (
    analysis_df.groupby(["example_id", "mode"], dropna=False)
    .agg(
        states=("state_id", "nunique"),
        resistance_available=("Lambda_matched", lambda s: np.isfinite(s).sum()),
        timing_available=(PRIMARY_Y_METRIC, lambda s: np.isfinite(s).sum()),
        primary_pairs=("primary_pair_complete", "sum"),
    )
    .reset_index()
)
display(coverage)
display(analysis_df["plot_category"].value_counts(dropna=False).rename("states").to_frame())
print(f"Technical promotion aliases retained in coverage: {analysis_df.technical_alias.sum()}")
display(analysis_df.groupby('example_id').domain_expanded.sum().rename('expanded_domains').to_frame())
display(analysis_df.event_duration_status.value_counts().rename('states').to_frame())

# No silent fallback to sampled peaks when interpolation is unresolved.
display(analysis_df.peak_interpolation_status.value_counts().rename("interpolation status"))


## Define the plotted population

The optional edge-compound sensitivity filter removes compound objects at the first and last compound downstream-order positions within each example. Its current setting is shown by EXCLUDE_NETWORK_EDGE_COMPOUNDS above; all figures and curve fits below use that same selection.

Deduplication precedes the edge filter. Shared child geometry with a different comparison domain remains a distinct comparison; equal plot coordinates alone are never deduplicated. Percentage lag, absolute timing, duration and peak excess use a common valid cohort. The full alias table preserves selection identities.


In [ ]:
from rapid_tools.comparison_population import plotted_comparisons

# Canonicalize before applying display filters: aliases cannot bypass exclusions.
plot_df, comparison_aliases = plotted_comparisons(
    analysis_df, exclude_edge_compounds=EXCLUDE_NETWORK_EDGE_COMPOUNDS)
print(f"Plotting {len(plot_df)} distinct comparisons from {plot_df.example_id.nunique()} examples.")
print(f"Full-inventory duplicate selection rows removed: {(~comparison_aliases.is_retained).sum()}")
display(plot_df.plot_category.value_counts().rename('Distinct plotted comparisons').to_frame())
display(plot_df.loc[plot_df['mode'].eq('island')].groupby('hydraulic_domain_type').size().rename('Island-only comparison context').to_frame())
# All original selections are retained in comparison_aliases and selection_aliases.


## Primary resistance–timing result

The first three views show the signed response, log y magnitude, and log x/y magnitudes. The x-axis is L_f × E × [−log10(Λ_K)]; the y-axis is refined peak-lag change (%), normalized by parent modeled lag. The immediately following section repeats the four timing views for absolute hours. Log axes show absolute magnitudes, with exact zeros at labelled display-only positions.

Each plot has three stacked legends beside it: experiment category, sign of −log10(Λ_K), and lag sign. Each sign legend offers +, − and 0, and all three filters intersect. Λ_K itself is a positive resistance ratio; its transformed value −log10(Λ_K) is positive for resistance reduction, negative for resistance increase, and zero for no change. Category colours stay consistent across percentage timing, absolute timing and speed plots. Linear views use circles; magnitude views use circles for resistance reduction and an earlier peak (or higher speed), and diamonds for other signs.

The three speed views now show signed **absolute speed change (m/s)**, with magnitudes on logarithmic axes. Positive means faster; negative means slower. A fourth lag plot and a fourth speed plot use X_raw = L_f E Λ_K with log-scaled x and y axes. This removes the inner logarithm only. The resistance-sign legend still describes −log10(Λ_K), not the sign of the positive raw ratio.


In [ ]:
from importlib import reload
import rapid_tools.response_diagnostics as response_diagnostics
reload(response_diagnostics)
from rapid_tools.response_diagnostics import make_signed_magnitude_scatter, show_signed_scatter

# Three matching views: signed linear, log magnitude y, and log magnitude x/y.
lag_figures = []
for log_x, log_y, view in [
    (False, False, 'signed axes'),
    (False, True, 'log y'),
    (True, True, 'log x and log y'),
]:
    lag_fig = make_signed_magnitude_scatter(
        plot_df, x=PRIMARY_X_METRIC, y=PRIMARY_Y_METRIC,
        log_x=log_x, log_y=log_y, response_name='Lag', y_label='Refined lag change (%)',
        title=f'Resistance–percentage timing change: {view}')
    lag_figures.append(lag_fig)
    show_signed_scatter(lag_fig)
fig, negative_lag_fig, x_log_fig = lag_figures

# Fourth view: remove only the inner logarithm; retain the length/exposure factors.
RAW_X_METRIC = 'length_exposure_lambda'
RAW_X_LABEL = '<i>L</i><sub>f</sub> · <i>E</i> · Λ<sub>K</sub>'
plot_df[RAW_X_METRIC] = plot_df['length_fraction'] * plot_df['width_fraction'] * plot_df['Lambda_matched']
raw_lambda_lag_fig = make_signed_magnitude_scatter(
    plot_df, x=RAW_X_METRIC, y=PRIMARY_Y_METRIC,
    log_x=True, log_y=True, response_name='Lag', y_label='Refined lag change (%)', x_label=RAW_X_LABEL,
    title='Percentage timing change: raw Λ_K inside predictor; log x and log y')
show_signed_scatter(raw_lambda_lag_fig)
lag_figures.append(raw_lambda_lag_fig)


## Primary resistance–absolute timing result

This repeats the preceding **four timing plots** with the same states, x variables, category colours and sign filters. Only the y variable changes:

$$Y_h=\frac{\hat t_{c,\mathrm{out}}-\hat t_{p,\mathrm{out}}}{3600},\qquad
Y_\%=100\frac{\hat t_{c,\mathrm{out}}-\hat t_{p,\mathrm{out}}}{\hat t_{p,\mathrm{out}}-t_{\mathrm{in}}}.$$

Times in these equations are seconds; hats denote refined outlet peaks. **Absolute** means an unnormalized time difference, not an unsigned value: negative hours mean an earlier child peak, positive hours mean later. The log views show magnitude and retain selectable signs. These are direct modeled hours, not observed-gauge-scaled hours. The absolute difference avoids the parent-lag denominator but can still depend on the surrounding network and hydraulic settings.


In [ ]:
absolute_lag_figures = []
for log_x, log_y, view in [
    (False, False, 'signed axes'),
    (False, True, 'log y'),
    (True, True, 'log x and log y'),
]:
    absolute_lag_fig = make_signed_magnitude_scatter(
        plot_df, x=PRIMARY_X_METRIC, y=ABSOLUTE_Y_METRIC,
        log_x=log_x, log_y=log_y, response_name='Lag',
        y_label='Refined child − parent peak timing (h)',
        title=f'Resistance–absolute timing difference: {view}')
    absolute_lag_figures.append(absolute_lag_fig)
    show_signed_scatter(absolute_lag_fig)

raw_lambda_absolute_lag_fig = make_signed_magnitude_scatter(
    plot_df, x=RAW_X_METRIC, y=ABSOLUTE_Y_METRIC,
    log_x=True, log_y=True, response_name='Lag', x_label=RAW_X_LABEL,
    y_label='Refined child − parent peak timing (h)',
    title='Absolute timing difference: raw Λ_K inside predictor; log x and log y')
show_signed_scatter(raw_lambda_absolute_lag_fig)
absolute_lag_figures.append(raw_lambda_absolute_lag_fig)

assert plot_df[[PRIMARY_Y_METRIC, ABSOLUTE_Y_METRIC]].notna().all().all()
print(f"Both timing sections use the same {len(plot_df)} comparisons from {plot_df.example_id.nunique()} examples.")


## Exact axes, domain selection, and the two logarithms

For each state, let $D$ be its selected hydraulic comparison domain. A hydraulically closed selected unit uses its own links (`closed_unit`). A closed compound selection uses all its selected links (`closed_compound_unit`). If the selected unit is not closed, the code expands to the complete directed domain between its enclosing compound's outer terminals (`containing_compound_context`); closure is checked and unresolved domains raise an error. This is a deterministic rule, not an optional interpretation.

With `LENGTH_MODE = "matched_parent_vs_network"`, the numerator is the sum of **base-network link lengths in D**, counted once, and the denominator is the sum of link lengths across the entire base network:

$$f_L=\frac{\sum_{i\in D}\ell_i}{\sum_{i\in\mathrm{base\ network}}\ell_i},\qquad
\Lambda_K=\frac{K_{\mathrm{after},D}}{K_{\mathrm{before},D}},\qquad
X=f_L\,\underbrace{E}_{\mathrm{exposure}}\,[-\log_{10}(\Lambda_K)].$$

**Exposure is explicitly included.** $E=1$ for `closed_compound_unit` and `containing_compound_context`. For a closed unit/nested domain, $E$ is the routed base-network width share: the reference inflow share is split at each bifurcation in proportion to outgoing branch widths, summed at reconvergences, and accumulated upon first entry into the target. It is a width-derived discharge-exposure proxy, not a measured discharge fraction or a global sum-of-widths fraction.

The inner logarithm compares multiplicative resistance changes: $\Lambda_K=0.1,1,10$ yields $-\log_{10}\Lambda_K=1,0,-1$. The ratio already removes absolute resistance scale; its logarithm additionally treats reciprocal changes symmetrically and converts products of ratios into sums. This is a modelling choice, not an obligation for ratios. Near no change, $-\log_{10}\Lambda_K\simeq(1-\Lambda_K)/\ln 10$. Far from one, it gives a much larger range than $1-\Lambda_K$, which is bounded above by one for positive resistance. Thus it expresses fold change rather than absolute resistance removed. A fitted relationship must justify this choice; merely looking straight on log axes does not.

Let $t_{\mathrm{out},s}^{\mathrm{peak}}$ and $t_{\mathrm{in},s}^{\mathrm{peak}}$ be the selected outlet and inflow peak times for state $s$. The implementation computes

$$\tau_s=t_{\mathrm{out},s}^{\mathrm{peak}}-t_{\mathrm{in},s}^{\mathrm{peak}},\qquad
r=\frac{\tau_s}{\tau_0}-1,\qquad Y=100r.$$

Here $0$ is the corresponding base RAPID run. Negative $Y$ means an earlier peak relative to inflow; positive means later. The denominator is **base RAPID peak lag**, not the observed gauge lag. A zero/undefined base lag has no defined percentage response. Outlet peak times now use guarded three-point quadratic interpolation around the saved selected maximum. The common input peak reference remains the saved forcing peak. Boundary, tied, nonconcave or out-of-bin estimates are excluded; the same valid population is used in the percentage and absolute sections. Interpolation reduces timestamp-bin ties but does not establish physical or observational timing accuracy. Sampled timing columns are retained for diagnostics.

The first logged figure uses coordinates $(X,\log_{10}|Y|)$. The double-log figure uses

$$\left(\log_{10}|X|,\log_{10}|Y|\right),\qquad
\log_{10}|X|=\log_{10}f_L+\log_{10}E+\log_{10}|-\log_{10}\Lambda_K|,$$

for strictly positive $f_L,E$ and nonzero $X,Y$. The outer log scales the **whole composite predictor**. It is not $-\log_{10}(\log_{10}\Lambda_K)$. Absolute values fold signs together visually; the original signs remain selectable and the fitted models below keep sign quadrants separate. Exactly zero values are shown one decade below the smallest nonzero magnitude, with an explicit `0 (display)` tick. That position is only a display convention, never a replacement value for scoring or fitting.

### Relative speed as the intermediate step to absolute speed

The old observed-gauge-scaled hour response is $\Delta\tau_{\mathrm{obs}}=r\,\tau_{\mathrm{obs}}$; the direct RAPID hour response is $r\tau_0/3600$. At fixed reference distance $L$, applying the former change gives $\tau_{\mathrm{obs},s}=(1+r)\tau_{\mathrm{obs}}$. Therefore either positive reference travel time gives the same relative apparent speed:

$$S=100\left(\frac{L/\tau_s}{L/\tau_0}-1\right)
=100\left(\frac{1}{1+Y/100}-1\right)
=-\frac{100Y}{100+Y}.$$

For example, a 20% shorter lag means a 25% higher apparent peak speed. This requires positive base and variant lags; zero/negative lags are undefined for this speed interpretation. It is a dimensionless change in **apparent peak-propagation speed**, not measured water velocity. Percentage lag was already dimensionless; this changes the physical interpretation, not the amount of independent information. It assumes the same reference distance before and after collapse.

Implementation for this refresh: `rapid_tools/refreshed_analysis.py`; the selected peak bin comes from production `hydrograph_metrics.json`, and refined outlet timing is extracted from the saved `outlet_hydrograph.csv`. Static selection metadata never supplies old numerical results. Case 41 expanded domains use their surviving boundary memberships and computed width-flow exposure. Position/order diagnostics use longest physical downstream paths in the refreshed base graph; they do not define the length-fraction numerator.
The inner-log slope at fixed length/exposure is dX/dΛ = −f_L E / (Λ ln 10). On the outer log axis, points with Λ near one approach X = 0 and move far to the left; small numerical changes near no resistance change can therefore move points substantially in log space. Equal visual distances represent multiplicative differences in |X|, not equal changes in the original resistance ratio. The log transform should be interpreted with peak-time resolution and numerical precision in mind.

Here Y is the numerical percentage value. Using the fractional change r instead would shift log10|Y| by −2; it would change a fitted intercept but not a power-law slope.

### Absolute speed change and the raw-ratio alternative

The speed panels use $v_0=L_{ref}/(3600\tau_{obs,h})$ and $\Delta v=-[r/(1+r)]v_0$ in m/s. The length is measured **along the existing single-channel reference geometry between the projected upstream and downstream gauge positions**. It is fixed across all variants of an example; it is not the sum of network branch lengths. Gauge coordinates prefer the manually updated source used by the experiments. The reference table displays length, observed peak lag, reference speed, and gauge projection offsets. Missing or nonpositive reference lags produce no speed estimate. These are changes in apparent peak-propagation speed, not measured water velocity.

Unlike percentage speed, absolute speed introduces an example-specific observed speed scale. It can change cross-network comparisons, but within one example it still rescales the same timing response.

The added predictor is $X_{raw}=f_L E\Lambda_K$. Its log-axis coordinate is $\log_{10}f_L+\log_{10}E+\log_{10}\Lambda_K$, rather than $\log_{10}f_L+\log_{10}E+\log_{10}|-\log_{10}\Lambda_K|$. These are different predictors. At no resistance change ($\Lambda_K=1$), $X_{raw}=f_LE$, whereas the original X is zero. The raw predictor therefore represents a weighted resistance ratio rather than a weighted resistance reduction.


In [ ]:
from rapid_tools.speed_reference import build_speed_reference_frame, absolute_speed_change

# A fixed single-channel distance between gauge projections, paired with the
# observed downstream-minus-upstream peak lag, supplies a speed scale in m/s.
speed_reference = build_speed_reference_frame(EXAMPLE_IDS, output_root=OUTPUT_ROOT)
display(speed_reference[['example_id', 'reference_length_m', 'observed_lag_hours',
                         'reference_speed_mps', 'up_snap_distance_m',
                         'down_snap_distance_m', 'status']])
speed_plot_df = plot_df.copy()
speed_plot_df['example_id'] = speed_plot_df['example_id'].astype(str).str.zfill(2)
speed_plot_df = speed_plot_df.merge(speed_reference, on='example_id', how='left', validate='many_to_one')
SPEED_Y_METRIC = 'peak_speed_change_mps'
speed_plot_df[SPEED_Y_METRIC] = absolute_speed_change(
    speed_plot_df[PRIMARY_Y_METRIC], speed_plot_df.reference_length_m,
    speed_plot_df.observed_lag_hours)
# Refined percentages require positive parent lag; r > -1 also requires positive child lag.
speed_plot_df.loc[speed_plot_df[PRIMARY_Y_METRIC].le(-100), SPEED_Y_METRIC] = np.nan
print(f"Absolute speed available for {speed_plot_df[SPEED_Y_METRIC].notna().sum()} / {len(speed_plot_df)} states.")

speed_figures = []
for log_x, log_y, view in [
    (False, False, 'signed axes'),
    (False, True, 'log y'),
    (True, True, 'log x and log y'),
]:
    speed_fig = make_signed_magnitude_scatter(
        speed_plot_df, x=PRIMARY_X_METRIC, y=SPEED_Y_METRIC,
        log_x=log_x, log_y=log_y, response_name='Speed', y_label='Speed change (m/s)',
        title=f'Resistance–absolute speed change: {view}')
    speed_figures.append(speed_fig)
    show_signed_scatter(speed_fig)
raw_lambda_speed_fig = make_signed_magnitude_scatter(
    speed_plot_df, x=RAW_X_METRIC, y=SPEED_Y_METRIC,
    log_x=True, log_y=True, response_name='Speed', x_label=RAW_X_LABEL,
    y_label='Speed change (m/s)',
    title='Absolute speed change: raw Λ_K inside predictor; log x and log y')
show_signed_scatter(raw_lambda_speed_fig)
speed_figures.append(raw_lambda_speed_fig)


## Conditional curve comparison and upper-tail sensitivity

These tests use the same primary plotted population, including its configured edge-compound filter. They exclude exactly zero/nonfinite values and fit **each original-sign quadrant separately** with $u=|X|$, $M=|Y|$:

- Power law: $\log_{10}M=a+b\log_{10}u$, equivalently $M=10^a u^b$.
- Exponential: $\log_{10}M=a+bu$, equivalently $M=10^a\exp[(b\ln10)u]$.
- Quadratic in log space: $\log_{10}M=a+b\log_{10}u+c(\log_{10}u)^2$.

All three minimize log-space residuals, making the comparison consistent. Scores are pooled **leave-one-example-out** predictions, reported in both log10 and original magnitude space on the same observations. Back-transforms are direct, without a mean-bias correction. A quadrant with insufficient independent examples or a deficient training design receives no complete CV score. Fitted curves/coefficients use the full quadrant and are descriptive; their in-sample score is explicitly separate.

The tail sensitivity refits the power-law slope after removing the upper x decile. It tests slope stability over the observed range; it cannot reconstruct an unobserved tail or demonstrate a truncated power-law distribution. Zero changes may reflect finite peak-time resolution and should not be imputed into a power law. These are response-curve regressions, not tests that the distribution of observations has a power-law tail.

This differs from the prediction notebook: here the question is the functional relationship between **one composite predictor** and timing magnitude. The prediction grid additionally tests geometry/topology combinations and models the signed response. Both use network-grouped validation, but their populations and targets must match before comparing scores.

In [ ]:
from rapid_tools.response_diagnostics import fit_response_relationships
import matplotlib.pyplot as plt

curve_scores, curve_parameters, curve_predictions = fit_response_relationships(
    plot_df, x=PRIMARY_X_METRIC, y=PRIMARY_Y_METRIC)
display(curve_scores)
display(curve_parameters)

if not curve_parameters.empty:
    quadrants = curve_parameters.quadrant.unique()
    curve_fig, axes = plt.subplots(1, len(quadrants), figsize=(6 * len(quadrants), 5), squeeze=False)
    for axis, quadrant in zip(axes.ravel(), quadrants):
        points = curve_predictions.loc[curve_predictions.quadrant.eq(quadrant)].drop_duplicates('row_index')
        axis.scatter(points.x_magnitude, points.observed_magnitude, alpha=0.35, s=14)
        for row in curve_parameters.loc[curve_parameters.quadrant.eq(quadrant)].itertuples():
            u = np.geomspace(row.x_min, row.x_max, 150)
            z = u if row.model == 'exponential' else np.log10(u)
            with np.errstate(over='ignore'):
                magnitude = 10 ** (row.intercept_log10 + row.b*z + row.c*z*z)
            matched_score = curve_scores.loc[curve_scores.quadrant.eq(quadrant) & curve_scores.model.eq(row.model), 'r2_log10_cv']
            curve_label = row.model + (f' (CV log R²={matched_score.iloc[0]:.2f})' if len(matched_score) else '')
            axis.plot(u, magnitude, '--' if 'without' in row.model else '-', label=curve_label)
        axis.set(xscale='log', yscale='log', xlabel='|Length fraction × exposure × −log10(Λ_K)|',
                 ylabel='|Lag shift| [%]', title=quadrant)
        axis.legend(fontsize=8)
    plt.tight_layout()

## Historical fit results

The earlier September fit scores described the previous hydraulic policy and cohort. They are not validation results for the October refresh and are deliberately not repeated as current findings. Use the freshly executed tables above. Island coverage and the optional population filter must be stated alongside any reported score.


## Width dependence under the updated depth law

The refreshed routing and resistance share

$$D=D_{ref}(W/W_{ref})^{0.65},\qquad K_b=W/D=K_{b,ref}(W/W_{ref})^{0.35}.$$

The wide-channel Chezy edge resistance remains

$$K_{res}=\frac{\ell}{C_z^2 g W^2 D^3}.$$

With the reference anchors, length and Chezy coefficient held fixed, this now gives **$K_{res}\propto W^{-3.95}$**, not the fixed-$K_b$ exponent $-5$. Local uncapped routing celerity scales as **$c\propto W^{0.4333}$** at fixed slope, Manning n and reference anchors. Caps can interrupt that scaling. These are local constitutive scalings, not fitted laws for peak timing or event shape.

The settings are $n=0.035$, celerity caps 0.05–5 m/s, 200 m target segmentation, $X=0.1$, and the common depth exponent 0.65. Case 37 uses 94.375 s and case 46 uses 67.8125 s; other cases retain their selected shared clocks. Resistance retains Chezy coefficient 22. Routing $K=\ell_{effective}/c$ is a travel-time parameter, distinct from hydraulic resistance $K_{res}$.


## Is the fitted relationship informative beyond the constructed predictor?

Compare the composite predictor, length/exposure-weighted raw ratio and Lambda alone on the same current reduced-resistance/earlier-peak cohort. Keep held-out network groups and response definitions identical. The results below are recalculated from the refreshed runs; historical September scores are not reused. Multiple selections can share a routed state, so pooled row counts do not equal independent experiments.


In [ ]:
from rapid_tools.response_diagnostics import fit_response_relationships

comparison_population = plot_df.loc[(plot_df[PRIMARY_X_METRIC] > 0) & (plot_df[PRIMARY_Y_METRIC] < 0)].copy()
transform_scores = []
for predictor in [PRIMARY_X_METRIC, RAW_X_METRIC, 'Lambda_matched']:
    comparison_scores, _, _ = fit_response_relationships(
        comparison_population, x=predictor, y=PRIMARY_Y_METRIC)
    comparison_scores.insert(0, 'predictor', predictor)
    transform_scores.append(comparison_scores)
transform_scores = pd.concat(transform_scores, ignore_index=True)
display(transform_scores[['predictor', 'model', 'n', 'n_cv', 'r2_log10_cv', 'r2_linear_cv']])


## Timing and event-shape sensitivity

The accepted event duration is first departure beyond 1% of the parent's peak-to-initial amplitude to the final sustained return within 1% of the parent's peak-to-final amplitude. The same absolute tolerances are used for each child. Crossings are interpolated; final settlement must last at least one hour, and censored events are flagged rather than imputed. This also captures a later secondary peak above tolerance.

Peak-excess change is `100 × [(child peak − child initial plateau)/(parent peak − parent initial plateau) − 1]`. Negative values mean reduced event peak height. It differs from total peak-discharge change when baseflow is substantial. Both duration and peak-excess changes are signed percentages. The remaining timing panels are supplementary diagnostics.


In [ ]:
# TIMING_SENSITIVITY_METRICS = [
#     "event_duration_change_pct",
#     "peak_excess_change_pct",
#     ABSOLUTE_Y_METRIC,
#     "peak_magnitude_pct_change_vs_base",
#     "centroid50_shift_hours",
#     PRIMARY_Y_METRIC,
#     "peak_speed_change_mps",
#     "delta_rise_time_hours",
#     "delta_fall_time_50_hours",
#     "delta_fall_time_10_hours",
#     "delta_e_folding_time_hours",
#     "peak_window_width95_shift_hours",
# ]

# for y_metric in TIMING_SENSITIVITY_METRICS:
#     print(f"Plotting timing sensitivity metric: {y_metric}")
#     metric_frame = speed_plot_df if y_metric == "peak_speed_change_mps" else plot_df
#     if y_metric not in metric_frame or not pd.to_numeric(metric_frame[y_metric], errors="coerce").notna().any():
#         print(f"Skipping unavailable metric: {y_metric}")
#         continue
#     make_timing_scatter(
#         metric_frame,
#         x=PRIMARY_X_METRIC,
#         y=y_metric,
#         color="plot_category",
#         symbol="mode",
#         hover_fields=["example_id", "target_id", "state_id"],
#     ).show()

## Resistance/geometry formulation sensitivity

This block shows how the relationship changes as resistance is considered alone and then weighted by spatial extent and routed width share.

In [ ]:
# X_SENSITIVITY_METRICS = [
#     "Lambda_matched",
#     "neg_log10_lambda",
#     'width_fraction',
#     "length_weighted_log_lambda",
#     "len_fraction_width_weighted_log_lambda",
#     "child_parent_weighted_log_lambda",
#     "child_parent_width_weighted_log_lambda",
# ]

# for x_metric in X_SENSITIVITY_METRICS:
#     if x_metric not in plot_df or not pd.to_numeric(plot_df[x_metric], errors="coerce").notna().any():
#         print(f"Skipping unavailable metric: {x_metric}")
#         continue
#     make_timing_scatter(
#         plot_df,
#         x=x_metric,
#         y=PRIMARY_Y_METRIC,
#         color="plot_category",
#         symbol="mode",
#         hover_fields=["example_id", "target_id", "state_id"],
#     ).show()

## Association summary and influential states

Spearman correlation is reported as a descriptive association, not as an independent-observation significance test: states from the same river network share a base network and forcing. The final table exposes the largest timing responses for case-level review.

In [ ]:
# Correlation screen: rank pre-specified, physically interpretable pairs rather than
# selecting a relationship by visual appearance.  Spearman rho is robust to non-linear
# monotonic responses and unequal scaling between examples.
X_SCREEN_METRICS = [
    "Lambda_matched",                 # raw selected-domain Lambda_K (shown on a log x-axis below)
    "neg_log10_lambda",              # resistance reduction: positive means lower resistance
    "length_weighted_log_lambda",
    "width_weighted_log_lambda",
    "len_fraction_width_weighted_log_lambda",
]
Y_SCREEN_METRICS = [
    "event_duration_change_pct",
    "peak_excess_change_pct",
    PRIMARY_Y_METRIC,
    "lag_to_inflow_peak_hour_shift_vs_observed_peak",
    ABSOLUTE_Y_METRIC,
    "rise95_time_shift_hours",
    "centroid50_shift_hours",
    "delta_rise_time_hours",
    "delta_fall_time_50_hours",
    "delta_fall_time_10_hours",
    "delta_e_folding_time_hours",
    "peak_window_width95_shift_hours",
]
X_SCREEN_METRICS = [metric for metric in X_SCREEN_METRICS if metric in plot_df]
Y_SCREEN_METRICS = [metric for metric in Y_SCREEN_METRICS if metric in plot_df]

def spearman_summary(frame, x_metric, y_metric):
    pair = frame[[x_metric, y_metric]].apply(pd.to_numeric, errors="coerce").dropna()
    return {
        "n": len(pair),
        "spearman_rho": pair.corr(method="spearman").iloc[0, 1] if len(pair) >= 3 else np.nan,
    }

screen_rows = []
for x_metric in X_SCREEN_METRICS:
    for y_metric in Y_SCREEN_METRICS:
        for subset_name, subset in [("all", plot_df), ("indep", plot_df.loc[plot_df["mode"].eq("indep")]), ("compound", plot_df.loc[plot_df["mode"].eq("compound")]), ("island", plot_df.loc[plot_df["mode"].eq("island")])]:
            screen_rows.append({"subset": subset_name, "x_metric": x_metric, "y_metric": y_metric, **spearman_summary(subset, x_metric, y_metric)})
correlation_screen = pd.DataFrame(screen_rows)
display(correlation_screen.sort_values(["subset", "spearman_rho"], key=lambda s: s.abs() if s.name == "spearman_rho" else s, ascending=[True, False]))

# Stability check: a pooled association is reportable only when the per-example/mode
# correlations have a consistent sign.
stratified_rows = []
for x_metric in X_SCREEN_METRICS:
    for y_metric in Y_SCREEN_METRICS:
        for (example_id, mode), group in plot_df.groupby(["example_id", "mode"]):
            stratified_rows.append({"x_metric": x_metric, "y_metric": y_metric, "example_id": example_id, "mode": mode, **spearman_summary(group, x_metric, y_metric)})
stratified_correlation = pd.DataFrame(stratified_rows)
stability_screen = (
    stratified_correlation.dropna(subset=["spearman_rho"])
    .assign(sign=lambda d: np.sign(d["spearman_rho"]))
    .groupby(["x_metric", "y_metric"], as_index=False)
    .agg(groups=("spearman_rho", "size"), median_rho=("spearman_rho", "median"), min_rho=("spearman_rho", "min"), max_rho=("spearman_rho", "max"), sign_agreement=("sign", lambda s: max((s > 0).mean(), (s < 0).mean())))
    .sort_values(["sign_agreement", "median_rho"], key=lambda s: s.abs() if s.name == "median_rho" else s, ascending=False)
)
display(stability_screen)

# Scale sensitivity. Lambda_K is positive, so raw Lambda_K is best visualized on a
# logarithmic x-axis. All timing and shape shifts are signed, so a normal log y-axis
# is invalid; asinh is a sign-preserving, log-like display transform.
LOG_VIEW_Y = ABSOLUTE_Y_METRIC
if LOG_VIEW_Y in plot_df and (pd.to_numeric(plot_df["Lambda_matched"], errors="coerce") > 0).any():
    raw_lambda_view = make_timing_scatter(plot_df, x="Lambda_matched", y=LOG_VIEW_Y, color="plot_category", symbol="mode", hover_fields=["example_id", "target_id", "state_id", "hydraulic_domain_type"])
    raw_lambda_view.update_xaxes(type="log", title="Lambda_K (selected hydraulic domain; log scale)")
    raw_lambda_view.show()

for y_metric in [ABSOLUTE_Y_METRIC, "peak_window_width95_shift_hours"]:
    if y_metric not in plot_df:
        continue
    values = pd.to_numeric(plot_df[y_metric], errors="coerce")
    scale = values.abs().loc[values.ne(0)].median()
    if not np.isfinite(scale) or scale <= 0:
        continue
    transformed = f"{y_metric}_asinh"
    plot_df[transformed] = np.arcsinh(values / scale)
    view = make_timing_scatter(plot_df, x=PRIMARY_X_METRIC, y=transformed, color="plot_category", symbol="mode", title=f"asinh({y_metric} / {scale:.3g}) vs {PRIMARY_X_METRIC}")
    view.show()

## Negative current-inflow coefficients — completed-run audit

For an internal routed segment,

$$O^{n+1}=a I^{n+1}+b I^n+c O^n,$$
$$a=\frac{\Delta t-2KX}{\Delta t+2K(1-X)},\quad
b=\frac{\Delta t+2KX}{\Delta t+2K(1-X)},\quad
c=\frac{2K(1-X)-\Delta t}{\Delta t+2K(1-X)}.$$

A negative **a** occurs when $\Delta t<2KX$. It is a negative algebraic weight on current inflow, **not negative inflow or discharge**. All three terms determine outflow. Inflow/outflow differences represent storage change even when every coefficient is positive. Here $a+b+c=1$, so steady inflow remains steady. External source segments use held forcing multiplied by $a+b$; the audit excludes their first segment from the count of actually applied negative current-inflow coefficients.

Allowing negative a removes a sufficient nonnegative-weight condition. Abrupt inputs can then produce small oscillations or negative responses; bounded c alone does not prove positivity for every possible forcing. The accepted safeguards retain $0\le c<1$, fixed hydraulics across clocks, timestep-convergence checks, and native-substep finite/nonnegative discharge checks without clipping. The audit below covers the actual completed events, not arbitrary forcing.

Original base links provide a non-repeated spatial count. All-state link/segment totals are **occurrences across runs**, not distinct physical river links. An original link can contain several routed 200 m segments.


In [ ]:
# Completed-run audit: coefficients are constant per segment within a run.
import json
AUDIT_ROOT = REPO_ROOT / "RAPID/validation/hydraulic_policy_20261003/completed_coefficients_20261005"
coefficient_summary = json.loads((AUDIT_ROOT / "summary.json").read_text())
assert Path(coefficient_summary['source']).resolve() == REFRESH_ROOT.resolve()
print(json.dumps(coefficient_summary, indent=2))
coefficient_states = pd.read_csv(AUDIT_ROOT / 'states.csv', dtype={'example': str})
coefficient_bases = pd.read_csv(AUDIT_ROOT / 'base_networks.csv', dtype={'example': str})
coefficient_bases['negative_link_pct'] = 100 * coefficient_bases.negative_links / coefficient_bases.n_links
display(coefficient_bases[['example', 'dt_s', 'n_links', 'negative_links', 'negative_link_pct',
                           'n_segments', 'negative_internal_segments', 'minimum_a', 'minimum_substep_flow_cms']])
display(coefficient_states.groupby('example').agg(
    distinct_runs=('fingerprint', 'nunique'),
    runs_with_negative_coefficients=('negative_links', lambda x: x.gt(0).sum()),
    minimum_native_flow=('minimum_substep_flow_cms', 'min')))


## Deduplicated four-response benchmark — 5 October 2026

**Reference: DEDUPLICATED-RESPONSES-2026-10-05.** Compare refined percentage lag, absolute timing (hours), event-duration change and peak-excess change using fixed geometric candidates. Real rivers are held out in turn; the test river never enters fitting or scaling. Ridge and shallow boosting retain the earlier model settings, without tuning.

Two populations are reported explicitly. **Full plotted** matches the figures above and supports scalar resistance/length/exposure predictors. **Common geometry** additionally requires the existing path descriptors (single-source/sink domain, at most 10,000 paths). Every response and candidate within a population uses identical rows. Do not attribute differences between these populations to a predictor improvement.

MAE is in response units: hours for absolute timing and percentage points for the other responses. **MAE skill = 1 − MAE / baseline MAE**, where each held-out river's baseline is the response mean of its training rivers. Positive skill beats that baseline. R² is also reported, including negative values. Equal-river summaries complement pooled, row-weighted results. Comparing candidate scores is exploratory model selection, not independent confirmation of a final formula.

Regenerate with `python RAPID/investigations/deduplicated_response_benchmark.py`. The previous benchmark is retained in its dated output directory; the tables below use the new deduplicated, matched-response results.


In [ ]:
import json
from IPython.display import Image
BENCH = REPO_ROOT / 'RAPID/validation/deduplicated_response_benchmark_20261005'
bench_metadata = json.loads((BENCH/'provenance.json').read_text())
assert EXCLUDE_NETWORK_EDGE_COMPOUNDS == bench_metadata['exclude_edge_compounds'], "Regenerate benchmark for the chosen edge filter."
benchmark_population = pd.read_parquet(BENCH/'plotted_population.parquet')
comparison_key = bench_metadata['comparison_key']
assert set(map(tuple, plot_df[comparison_key].to_numpy())) == set(map(tuple, benchmark_population[comparison_key].to_numpy())), "Benchmark and plot populations differ; regenerate."
print('Full plotted:', bench_metadata['plotted_comparisons'], 'Common geometry:', bench_metadata['common_geometry_comparisons'])
display(pd.read_csv(BENCH/'population_counts.csv'))
excluded_geometry = pd.read_csv(BENCH/'excluded_extended_geometry.csv', dtype={'example_id':str})
display(excluded_geometry.groupby('example_id').size().rename('Excluded from common geometry').to_frame())
benchmark_scores = pd.read_csv(BENCH/'scores.csv')
pooled = benchmark_scores.query("scope == 'pooled'")
display(pooled.pivot(index=['cohort','feature_set','model'], columns='response', values=['r2','mae','mae_skill']))
display(Image(filename=str(BENCH/'response_skill.png')))


### River and collapse-category checks

The predicted-versus-simulated panels use `selected_six` with boosting for all four responses; this fixed display is not a claim that it is the winning model. Scores by river and category expose pooled results dominated by a few rivers or repeated within-river conditions. “Island-only collapse” is retained only when no equivalent topological comparison exists.


In [ ]:
display(Image(filename=str(BENCH/'held_out_predictions.png')))
river_scores = benchmark_scores.query("scope == 'example_id'")
equal_river = river_scores.groupby(['cohort','feature_set','model','response']).agg(
    rivers=('group','nunique'), median_river_r2=('r2','median'),
    mean_river_mae_skill=('mae_skill','mean'), rivers_beating_baseline=('mae_skill', lambda x: int(x.gt(0).sum())))
display(equal_river)
display(benchmark_scores.query("cohort == 'common_geometry' and model == 'boosted' and feature_set in ['omega','selected_six'] and scope == 'plot_category'")
        .pivot(index=['feature_set','group'], columns='response', values=['r2','mae_skill']))


### What remains around omega?

Residual = simulated response − held-out prediction. These diagnostics use the same omega-only boosted model and common geometry population for each response. The heatmap shows median **within-river** rank associations with signed residuals. The underlying tables also contain pooled/category associations and associations with absolute residual size. These are clues for investigation; correlated features and model limitations prevent a causal reading.

The candidate-addition scores above test whether those descriptors actually improve prediction of unseen rivers. High-residual cases are listed for targeted hydrograph and numerical review, rather than automatically attributed to geometry.


In [ ]:
display(Image(filename=str(BENCH/'residual_geometry.png')))
residual_associations = pd.read_csv(BENCH/'residual_associations.csv', dtype={'group':str})
display(residual_associations.query("scope == 'example_id' and residual_kind == 'residual'")
        .groupby(['response','feature']).agg(rivers=('spearman','count'), median_rho=('spearman','median'),
        min_rho=('spearman','min'), max_rho=('spearman','max')))
high_residuals = pd.read_csv(BENCH/'high_residual_comparisons.csv', dtype={'example_id':str})
display(high_residuals[['response','example_id','state_id','plot_category','observed','predicted','residual']])


### Synthetic comparison with matched responses

Synthetic outlet peaks are now extracted with the **same guarded interpolation** used for real networks. The common input-peak reference within each synthetic pair is checked after removing its spin-up offset. Exact run-pair/domain duplicates are removed, and the existing synthetic development/geometry-holdout split is preserved. Duration retains the accepted 1% plateau rule and peak excess uses the shared plateau. No simulations are rerun.

Models are fitted on synthetic development cases with the fixed settings, evaluated on the untouched synthetic geometry holdout, and then applied to the real common-geometry cohort without real-data fitting. This separates response-definition differences from domain transfer. Grid ranges, forcing, allocation, hydraulics and routing assumptions still differ; a shared response definition alone does not establish physical generality. The feature-range table makes extrapolation visible.


In [ ]:
print(bench_metadata['synthetic_peak_audit'])
transfer_scores = pd.read_csv(BENCH/'synthetic_transfer_scores.csv')
display(transfer_scores.query("scope == 'pooled'").pivot(index=['feature_set','response'], columns='validation', values=['r2','mae','mae_skill']))
display(pd.read_csv(BENCH/'feature_ranges.csv'))


### Reading these results — 5 October 2026

On the **common 1,128-comparison cohort**, boosted omega predicts percentage lag with held-out R² **0.682**, while the selected six predict absolute hours with R² **0.544** (omega: **0.108**). Both beat their training-mean baseline in 15/16 rivers for their respective response. These are different prediction targets, so R² alone does not decide which timing metric is scientifically preferable.

**Population matters:** omega percentage-lag R² is **0.334** on all 1,652 plotted comparisons. The 524 excluded path-descriptor cases cannot be silently dropped from the overall conclusion. **Model form matters:** selected-six absolute timing has linear R² only **0.035**.

Duration and peak excess remain poorly predicted across rivers. Small positive R² from the largest feature set comes with negative pooled MAE skill. Residual geometry associations are exploratory; example 46 accounts for most of the largest absolute-timing and shape residuals.

The matched synthetic transfer here uses the **original full grid**, not the completed wider embedding extension. Every real length fraction lies below this original synthetic development range. Strong synthetic holdout scores do not transfer to real cases, so the extension and residual checks remain necessary before freezing a final predictor.

See the [dated findings and reproduction record](../validation/deduplicated_response_benchmark_20261005/README.md) for scores, definitions, limitations and next decisions. Reference: **DEDUPLICATED-RESPONSES-2026-10-05**.


### Which additions help in synthetics and real rivers? · 5 October 2026

Fixed boosted models, identical rows within each dataset, with no additional model tuning. This compares **predictive value of adding a group to omega**, not independent causal importance. Synthetic geometry holdout and real leave-river-out are different generalization tasks; their R² values are not directly comparable measures of difficulty.

For percentage lag, width dominance improves synthetic R² from .721 to .903, size/proportions to .873 and length–width association to .861. In real cases those additions give .680, .676 and .687 versus .682 for omega alone. For absolute timing, size helps in both (.203→.762 synthetic, .108→.368 real), and slope helps in both (.203→.314 synthetic, .108→.409 real). Real shape-response predictions remain weak despite useful synthetic shape predictions.

The controlled grid, different parameter ranges, stronger responses and simpler structures are plausible contributors. This comparison does not isolate their individual causes. A broader model search may help, but use nested river-grouped validation (inner model selection, outer assessment), retain simple baselines, and report equal-river scores. Choosing the best model on the current test scores would make the reported score optimistic.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
ADDITION_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
ADDITION_OUT = ADDITION_ROOT/'RAPID/validation/deduplicated_response_benchmark_20261005'
addition_synthetic = pd.read_csv(ADDITION_OUT/'synthetic_predictor_additions.csv')
addition_real = pd.read_csv(ADDITION_OUT/'scores.csv').query("cohort == 'common_geometry' and scope == 'pooled' and model == 'boosted'")
addition_real = addition_real[addition_real.feature_set.isin(addition_synthetic.feature_set)]
addition_both = pd.concat([addition_synthetic.assign(setting='Synthetic geometry holdout'), addition_real.assign(setting='Real river holdout')])
display(addition_both.pivot(index=['feature_set','setting'], columns='response', values='r2').round(3))


## Metric inventory, all-network lags and response magnitudes · 5 October 2026

The detailed geometry cohort excludes **524 path-rich comparisons**, all from examples 15 (215) and 22 (309), because of the 10,000-path enumeration cutoff. All 16 rivers remain, but their representation changes. Treat the full 1,652-comparison scalar benchmark as the broader result; compare feature sets only on identical rows.

The table below distinguishes gauge/base-model lag from the child-minus-parent collapse effects. Model lags were reread from production hydrographs rather than copied from the older review summary. Shapes change very little: median absolute duration and excess changes are 0.00091% and 0.00291%, respectively. Low R² alone is therefore insufficient to judge whether prediction error matters. Duration is a whole-event measure, not peak width.

[Full dated inventory and per-network exports](../validation/metric_inventory_20261005/README.md).


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
METRIC_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
METRIC_OUT = METRIC_ROOT/'RAPID/validation/metric_inventory_20261005'
display(pd.read_csv(METRIC_OUT/'cohorts.csv', dtype={'example_id': str}))
network_lags = pd.read_csv(METRIC_OUT/'network_lags.csv', dtype={'example': str})
display(network_lags[['example','observed_first_peak_lag_h','model_interpolated_lag_h','lag_difference_h','model_downstream_peak_error_h']].round(2))
response_sizes = pd.read_csv(METRIC_OUT/'response_magnitudes.csv', dtype={'example': str})
display(response_sizes[response_sizes.example.eq('all')][['metric','median_absolute','p95_absolute','max_absolute']].round(6))
display(response_sizes[response_sizes.example.ne('all') & response_sizes.metric.isin(['interpolated_lag_shift_pct','timing_change_minutes'])][['example','metric','n','signed_min','signed_median','signed_max']].round(3))


## Metric equations used in the current benchmark

Notation: edge e has physical length l_e and width w_e; p denotes the matched parent domain, c its child, and N the full parent network. Path j joins the domain's source to its sink. Shared edges can occur in several paths. Natural logarithms are used for geometry-change ratios; resistance log uses base 10.

### Length, exposure and resistance

- Domain total channel length: L_sum,p = Σ_(e in p) l_e. Full network channel length L_sum,N = Σ_(e in N) l_e. These count each edge once, including parallel branches; neither is a single centerline length.
- Length fraction: f_L = L_sum,p / L_sum,N. Child/parent total-length ratio: r_L = L_sum,c / L_sum,p (available diagnostic, not the main f_L).
- Width-allocation edge proportion: a_e = w_e / Σ_(k outgoing from u) w_k. Propagate inlet share q_inlet=1 and q_v = Σ_(e entering v) q_u a_e. Raw domain exposure E_raw = q_source × Σ_(e leaving source inside p) w_e / Σ_(all e leaving source) w_e.
- Current exposure E: **1 for closed_compound_unit and containing_compound_context**, otherwise E_raw. This is the current domain-aware convention; E must not always be described as the actual fraction of inlet discharge entering a subdomain.
- Anchored depth: h_e = h_ref (w_e/W_ref)^0.65; local k_b,e = w_e/h_e. BASED supplies the retained reference width/depth. Cross-sectional area A_e=w_e h_e and wide-channel hydraulic radius R_h,e=h_e.
- Current Chezy edge resistance: K_e = l_e / (A_e² C_z² g R_h,e), with the repository's dimensionless C_z convention (default 22, g=9.81). Head-flow law ΔH_e=K_e Q_e |Q_e|. This K_e is not RAPID's travel-time K.
- Equivalent resistance: K_eq=ΔH_domain/Q_domain², computed by the nonlinear network solve. Series: Σ K_e; parallel: (Σ K_e^(-1/2))^(-2). General compounds require the solve.
- Resistance ratio λ=K_eq,c/K_eq,p; resistance log R=-log10(λ); main combined axis **ω=f_L E R**. Separate components in the benchmark are R, f_L and E.

### Path geometry (parent and child separately)

For P enumerated paths:

L_j = Σ_(e in j) l_e,
W_j = L_j / Σ_(e in j)(l_e/w_e),
a_j = W_j / Σ_j W_j.

W_j is a length-weighted harmonic width along path j. The path-width shares a_j are not the recursively allocated discharge shares q_v above.

- Arithmetic mean path length: L_bar = (1/P) Σ_j L_j.
- Equivalent length: L_eq = Σ_j a_j L_j = Σ_j W_j L_j / Σ_j W_j.
- Equivalent width: W_eq = Σ_j W_j. This is a path-based aggregate; shared edges are counted on each enumerated path.
- Elongation: e = L_eq/W_eq = (Σ_j W_j L_j)/(Σ_j W_j)².
- Width dominance: D = max_j(W_j)/Σ_j W_j = max_j a_j.
- Path-length coefficient of variation: CV_L = sqrt[(1/P) Σ_j(L_j-L_bar)²]/L_bar. It is unweighted and uses population standard deviation.
- Signed length–width association: A_LW = L_eq/L_bar - 1. The older uncentered L_wa equals A_LW+1. Equal path lengths give A_LW=0 and L_wa=1. Positive association means width weighting favors longer paths; zero does not prove all lengths are equal.
- Domain reference slope: S_ref=(H_source-H_sink)/min_j L_j. This is head drop divided by shortest physical domain path, not mean edge slope.
- Surrounding lengths: L_up = shortest physical path from network inlet to parent-domain source; L_down = shortest physical path from parent-domain sink to network outlet.

### Parent–child changes and combinations

Δlog L_eq=ln(L_eq,c/L_eq,p); Δlog e=ln(e_c/e_p); ΔCV_L=CV_L,c−CV_L,p; ΔD=D_c−D_p.

The selected six are **R, E, L_eq,p, D_p, A_LW,p, S_ref,p**. Size/proportions means **L_eq,p and e_p**. Context means **L_up and L_down**. Change/context adds the four changes above and both surrounding lengths. Product candidates from the earlier search were ωD, ωA_LW and ωS_ref; they are not the current selected-six model.

Resistance and anchored depth are hydraulic-geometry descriptors, not unqualified geometry-only measurements. The current path benchmark does not include entropic braiding index as a fitted feature. Width dominance is not that index. The repository contains other exploratory descriptors; this inventory covers the current benchmark and its retained product candidates, not every historical notebook column.

## Current response equations

Let t_i be the sampled applied-input peak; t_p,t_c the guarded three-point interpolated parent and child outlet peaks. Parent transit lag τ_p=t_p−t_i. Common forcing implies child-minus-parent lag equals t_c−t_p.

- **Percentage lag change:** y_lag=100(t_c−t_p)/(t_p−t_i). This is not normalized by an absolute timestamp or by rise duration.
- **Absolute timing change:** y_time=(t_c−t_p)/3600 in hours (×60 for minutes). Negative means an earlier child peak.
- **Event duration:** initial/final plateau levels B_0,s and B_1,s are taken from each hydrograph. The parent defines shared tolerances ε_0=.01(Q_max,p−B_0,p), ε_1=.01(Q_max,p−B_1,p). For both states, t_start is the first departure beyond ε_0 from the initial plateau, and t_end is the final return within ε_1 of the final plateau, with at least one hour of final settlement. Crossings are linearly interpolated. D_s=t_end,s−t_start,s and y_duration=100(D_c/D_p−1). Later secondary peaks postpone the final return. **This is whole-event duration, not peak width/FWHM or a 1–99% cumulative-volume interval.**
- **Peak excess:** shared parent background B=max(B_0,p,B_1,p); P_p=Q_max,p−B, P_c=Q_max,c−B; y_peak=100(P_c/P_p−1). Negative means a lower child peak above that shared background. It is neither total-flow percentage change nor an attenuation-only metric.

For gauge validation only, τ_obs=t_down,obs−t_up,obs and δτ=τ_p−τ_obs. A direct downstream peak-date error is t_p,UTC−t_down,obs; it can differ from δτ because the applied forcing's peak need not coincide exactly with the sampled upstream-gauge maximum. Both are exported. Observed plateau/secondary-peak alternatives remain sensitivity cases, not silently substituted primary peaks.


## Bounded nested predictor search · 5 October 2026

Reference **NESTED-PREDICTOR-SEARCH-2026-10-05**. This extends predictor investigation after establishing that real shape changes are small. No routing or response definition changed.

All candidates use the same 1,128 eligible comparisons from 16 rivers. The 524 path-rich cases excluded from this cohort remain outside these results. Each outer fold holds out one complete river. Four grouped inner folds within the other rivers select among 12 candidates: four feature sets (omega; separate resistance/length fraction/exposure; selected six; selected six plus changes/context) crossed with linear Ridge, regularized quadratic Ridge and shallow histogram boosting. Scaling is fitted inside each training fold. Selection minimizes the mean of per-river MAEs, rather than pooling observations across unequally sized rivers. The outer river never enters this selection.

| Response | Outer pooled R² | Model MAE | No-change MAE | Rivers beating no-change |
|---|---:|---:|---:|---:|
| Percentage lag (percentage points) | 0.622 | 0.15372 | 0.29948 | 16/16 |
| Absolute timing (hours) | 0.544 | 0.13496 | 0.26578 | 15/16 |
| Event duration (percentage points) | -0.009 | 0.00684 | 0.00524 | 2/16 |
| Peak excess (percentage points) | -0.128 | 0.02088 | 0.01434 | 5/16 |

The timing models approximately halve pooled MAE relative to predicting no change. Absolute timing consistently selects the boosted selected-six model in all 16 outer folds; its MAE is about 8.1 minutes. Percentage lag selects quadratic omega in 10 folds and the extended boosted model in six. The nested search's R² of .622 does not improve on the prior fixed omega benchmark (.682); selecting by inner equal-river MAE need not maximize pooled outer R². This search does not justify additional complexity for the readable lag axis.

Both shape models lose to the no-change baseline, despite their very small errors in percentage-point units. Event duration selects a simple linear model in every fold. The result supports reporting effect-size distributions, resolution checks and the larger-response tail before seeking a more elaborate universal shape predictor. It does not prove exact zero physical change.

**Interpretation of small shape responses:** fixed routing coefficients produce a linear response to the imposed inlet hydrograph. Broad input features and common surrounding routing can make local changes appear mainly as small timing shifts. The real cases use saved upstream event hydrographs, which can contain multiple peaks, not necessarily an idealized single pulse. There is one imposed upstream input; unmodelled tributary event inputs and stage-dependent floodplain processes are absent. These are plausible mechanisms, not a quantified attribution of the small response. Event duration also does not measure every possible internal reshaping. A focused input-duration sensitivity with hydraulics held fixed could distinguish forcing dependence from a universally weak geometry effect.

**Validation limits:** candidate specifications reflect prior exploration of these same rivers. Nested selection prevents direct outer-fold tuning in this run, but does not make these rivers an untouched external validation set. Scores are exploratory, and comparison rows within a river are correlated. No broad hyperparameter optimization or synthetic model refit was performed.

Reproduce `RAPID/investigations/nested_predictor_search.py`. `provenance.json` fixes candidates and selection rules; `inner_scores.csv` retains every candidate/fold-selection score; `selections.csv` records each outer choice; `predictions.parquet` contains held-out predictions; `scores.csv` includes pooled/per-river results and zero/training-mean/training-median baselines. Fold assertions verify disjoint training/validation rivers.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
NESTED_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'RAPID').is_dir())
NESTED_OUT = NESTED_ROOT/'RAPID/validation/nested_predictor_search_20261005'
nested_scores = pd.read_csv(NESTED_OUT/'scores.csv')
nested_choices = pd.read_csv(NESTED_OUT/'selections.csv')
display(nested_scores[nested_scores.scope.eq('pooled')][['response','n','r2','mae','zero_mae','median_mae','skill_vs_zero']].round(5))
display(nested_scores[nested_scores.scope.eq('river')].groupby('response').agg(rivers_beating_zero=('skill_vs_zero',lambda v:int((v>0).sum())),median_river_r2=('r2','median')).round(3))
display(nested_choices.groupby(['response','feature_set','model']).size().rename('outer_folds_selected').to_frame())
